In [1]:
import numpy as np
import pandas as pd
import nltk
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from pathlib import Path
from dotenv import load_dotenv
import os

from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Auxiliares
def truncar_texto(texto, limite=100):

    texto = str(texto)

    if len(texto) <= limite:
        return texto

    return texto[:limite] + "..."


def vetor_medio_documento(tokens, model, vector_dim):

    vetores = [
        model.wv[token]
        for token in tokens
        if token in model.wv
    ]

    if len(vetores) == 0:
        return np.zeros(vector_dim)

    return np.mean(vetores, axis=0)

In [3]:
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")
    nltk.download("punkt_tab")


# Função Principal
def analisar_similaridade_w2v(
    df,
    colunas=["stop", "stop_lemma", "stop_stem"],
    top_n=30,
    vector_dim=100,
    window_size=5,
    min_word_count=1,
    sg=0,
    random_state=42
):

    for coluna in colunas:

        print("\n" + "=" * 90)
        print(f"ANALISANDO COLUNA: {coluna}")
        print("=" * 90)

        # Preparação
        dados = (
            df[coluna]
            .fillna("")
            .astype(str)
            .reset_index(drop=True)
        )

        dados = dados[dados.str.strip() != ""].reset_index(drop=True)

        if len(dados) < 2:
            print(f"Coluna {coluna} possui poucos dados.")
            continue

        documentos = dados.tolist()

        query = documentos[0]

        print("\nQUERY:")
        print(query)

        # Tokenização
        documentos_tokenizados = [
            word_tokenize(doc.lower())
            for doc in documentos
        ]

        # Words2Vec
        model_w2v = Word2Vec(
            sentences=documentos_tokenizados,
            vector_size=vector_dim,
            window=window_size,
            min_count=min_word_count,
            workers=4,
            sg=sg,
            seed=random_state
        )

        print(
            f"\nVocabulário: "
            f"{len(model_w2v.wv.index_to_key)} palavras"
        )

        # Vetores docs
        vetores_documentos = np.array([
            vetor_medio_documento(
                tokens,
                model_w2v,
                vector_dim
            )
            for tokens in documentos_tokenizados
        ])

        query_tokens = word_tokenize(query.lower())

        vetor_query = vetor_medio_documento(
            query_tokens,
            model_w2v,
            vector_dim
        )

        # Similaridade query x documentos
        similaridades = cosine_similarity(
            vetor_query.reshape(1, -1),
            vetores_documentos
        )[0]

        resultado = pd.DataFrame({
            "indice": range(len(documentos)),
            "documento": documentos,
            "similaridade": similaridades
        })

        # remove a própria query
        resultado = resultado[resultado["indice"] != 0]

        # Top mais similares
        top_similares = (
            resultado
            .sort_values(
                "similaridade",
                ascending=False
            )
            .head(top_n)
            .reset_index(drop=True)
        )

        # Menos similares
        top_menos = (
            resultado
            .sort_values(
                "similaridade",
                ascending=True
            )
            .head(top_n)
            .reset_index(drop=True)
        )

        # # HEATMAPS QUERY x DOCUMENTOS
        # fig = make_subplots(
        #     rows=1,
        #     cols=2,
        #     subplot_titles=(
        #         f"{top_n} MAIS SIMILARES",
        #         f"{top_n} MENOS SIMILARES"
        #     ),
        #     horizontal_spacing=0.08
        # )

        # z_top = np.array([
        #     top_similares["similaridade"].values
        # ])

        # z_bottom = np.array([
        #     top_menos["similaridade"].values
        # ])

        # # Top
        # fig.add_trace(

        #     go.Heatmap(
        #         z=z_top,

        #         x=top_similares["indice"],

        #         y=["similaridade"],

        #         customdata=np.array(
        #             top_similares["documento"]
        #         ).reshape(1, -1),

        #         text=np.round(
        #             top_similares["similaridade"],
        #             4
        #         ),

        #         texttemplate="%{text}",

        #         hovertemplate=
        #             "<b>Índice:</b> %{x}<br>" +
        #             "<b>Similaridade:</b> %{z:.4f}<br><br>" +
        #             "<b>Documento:</b><br>" +
        #             "%{customdata}<extra></extra>",

        #         colorscale="Blues"
        #     ),

        #     row=1,
        #     col=1
        # )

        # # Bottom
        # fig.add_trace(

        #     go.Heatmap(
        #         z=z_bottom,

        #         x=top_menos["indice"],

        #         y=["similaridade"],

        #         customdata=np.array(
        #             top_menos["documento"]
        #         ).reshape(1, -1),

        #         text=np.round(
        #             top_menos["similaridade"],
        #             4
        #         ),

        #         texttemplate="%{text}",

        #         hovertemplate=
        #             "<b>Índice:</b> %{x}<br>" +
        #             "<b>Similaridade:</b> %{z:.4f}<br><br>" +
        #             "<b>Documento:</b><br>" +
        #             "%{customdata}<extra></extra>",

        #         colorscale="Reds"
        #     ),

        #     row=1,
        #     col=2
        # )

        # # gráfico
        # fig.update_layout(
        #     title=f"QUERY x DOCUMENTOS — {coluna}",
        #     height=500,
        #     width=1600
        # )

        # fig.show()

        # Doc x Doc
        indices_heatmap = (
            top_similares["indice"].tolist()
            +
            top_menos["indice"].tolist()
        )

        indices_heatmap = list(
            dict.fromkeys(indices_heatmap)
        )

        docs_heatmap = [
            documentos[i]
            for i in indices_heatmap
        ]

        vetores_heatmap = vetores_documentos[
            indices_heatmap
        ]

        # Matriz similaridade
        matriz_sim = cosine_similarity(
            vetores_heatmap
        )

        labels = [
            f"{i}"
            for i in indices_heatmap
        ]

        # Heatmap Docs
        fig2 = px.imshow(
            matriz_sim,

            x=labels,
            y=labels,

            color_continuous_scale="Viridis",

            text_auto=".2f",

            aspect="auto"
        )

        customdata = []

        for linha_texto in docs_heatmap:

            linha_custom = []

            for coluna_texto in docs_heatmap:

                linha_custom.append([

                    truncar_texto(linha_texto, 100),

                    truncar_texto(coluna_texto, 100)
                ])

            customdata.append(linha_custom)

        customdata = np.array(customdata)

        fig2.update_traces(

            customdata=customdata,

            hovertemplate=
                "<b>Texto Linha:</b><br>%{customdata[0]}<br><br>" +
                "<b>Texto Coluna:</b><br>%{customdata[1]}<br><br>" +
                "<b>Similaridade:</b> %{z:.4f}<br>" +
                "<extra></extra>"
        )

        fig2.update_layout(
            title=(
                f"SIMILARIDADE DOCUMENTO x DOCUMENTO "
                f"({coluna})"
            ),
            height=1000,
            width=1000
        )

        fig2.show()

        # Tabelas
        print("\nTOP MAIS SIMILARES")
        display(
            top_similares[
                ["indice", "similaridade", "documento"]
            ]
        )

        print("\nTOP MENOS SIMILARES")
        display(
            top_menos[
                ["indice", "similaridade", "documento"]
            ]
        )

In [4]:
env_path = Path("..") / ".env"
load_dotenv(dotenv_path=env_path)

nome_arquivo = os.getenv("NOMALIZED_FILE_NAME")

caminho_csv = Path("..") / "data_output" / nome_arquivo

df = pd.read_csv(caminho_csv)

analisar_similaridade_w2v(
    df=df
)


ANALISANDO COLUNA: stop

QUERY:
oitava natal senhor jesus cristo dia circuncisão solenidade santa maria mãe deus concílio éfeso padres aclamaram theotókos porque nela verbo fez carne habitou homens filho deus príncipe paz dado nome acima todos nomes

Vocabulário: 16298 palavras



TOP MAIS SIMILARES


,indice,similaridade,documento
0,3242,0.999833,roma beata maria beltrame quattrócchi mãe famí...
1,4091,0.999788,são paulo cruz presbítero desde juventude dist...
2,438,0.999708,bordéus frança santa joana lestonnac ainda ado...
3,301,0.999687,viena áustria beato ladislau batthyány-strattm...
4,4486,0.999628,santa margarida nascida hungria casada malcom ...
5,70,0.999605,solenidade epifania senhor celebra tríplice ma...
6,1462,0.999502,festa santa catarina sena virgem doutora igrej...
7,2307,0.999476,santa isabel rainha portugal admirável interve...
8,3917,0.999455,memória senhora rosário dia recitação rosário ...
9,1024,0.999421,valdstena suécia santa catarina virgem filha s...



TOP MENOS SIMILARES


,indice,similaridade,documento
0,3898,0.837444,rímini marcas região itália beato alberto marv...
1,1632,0.837998,varennes território langres gália actualmente ...
2,2029,0.841698,auxerre gália lionense actualmente frança são ...
3,1550,0.842737,vienne gália lionense actual frança são nicéci...
4,3091,0.845720,metz gália bélgica actualmente frança são firm...
5,566,0.845965,château-landon gália actualmente frança são se...
6,3787,0.846142,auxerre gália lionense actual frança são frate...
7,2514,0.847096,auxerre gália lionense actualmente frança são ...
8,3271,0.847363,saintes gália actualmente frança são viviano b...
9,188,0.847572,gévaudan gália actualmente frança são firmino ...


...


ANALISANDO COLUNA: stop_stem

QUERY:
oit natal senh jesu crist dia circuncis solen sant mar mãe deu concíli éfes padr aclam theotók porqu nel verb fez carn habit hom filh deu príncip paz dad nom acim tod nom

Vocabulário: 12177 palavras



TOP MAIS SIMILARES


,indice,similaridade,documento
0,3917,0.999582,memór senh ros dia recit ros coro mari invoc a...
1,438,0.999542,bordéu franç sant joan lestonnac aind adolesc ...
2,3160,0.999398,memór virgem sant mar rainh deu luz filh deu p...
3,35,0.999398,sant nom jesu únic nom tud céu terr abism ajoe...
4,4091,0.999366,são paul cruz presbíter desd juventud distingu...
5,4936,0.999234,pass inumer sécul desd cri mund princípi deu c...
6,3548,0.999102,memór senh dor est pé junt cruz jesu associ ín...
7,854,0.999086,sant francisc roman religi dad cas aind adoles...
8,301,0.999059,vien áustr beat ladislau batthyány-strattmann ...
9,1462,0.999033,fest sant catarin sen virgem dou igrej tend to...



TOP MENOS SIMILARES


,indice,similaridade,documento
0,1632,0.769721,varenn territóri langr gál act franç são gengulf
1,1550,0.771943,vienn gál lionens act franç são nicéci bisp
2,2029,0.772539,auxerr gál lionens act franç são censúri bisp
3,1084,0.774590,senlil gál lugdun act franç são régul bisp
4,2589,0.775321,menat gál arven act franç são menel abad
5,188,0.776012,gévaudan gál act franç são firmin bisp
6,4068,0.777156,orang provenç reg gál act franç são florênci bisp
7,3091,0.777207,metz gál bélg act franç são firmin bisp
8,4633,0.777719,carpentr provenç act franç são sifr bisp
9,566,0.778342,château-landon gál act franç são severin abad ...
